# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">1. Model Training for New York City Taxi Trip Duration Dataset</p>

### Loading the necessary Libraries

In [1]:
# ==========================================
# 1. CORE DATA MANIPULATION & MATH
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# 2. VISUALIZATION
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# Render plots directly below notebook cells
%matplotlib inline

# ==========================================
# 3. SYSTEM CONFIGURATION
# ==========================================
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 4. DATA PREPARATION & MODEL SELECTION
# ==========================================
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ==========================================
# 5. MACHINE LEARNING MODELS (SCIKIT-LEARN)
# ==========================================
# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Tree-Based & Ensemble Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

# Other Algorithms
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# ==========================================
# 6. ADVANCED GRADIENT BOOSTING MODELS
# ==========================================
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping

# ==========================================
# 7. EVALUATION METRICS
# ==========================================
from sklearn.metrics import (
    r2_score, 
    mean_absolute_error, 
    mean_squared_error, 
    root_mean_squared_error
)

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">2. Read the Dataset</p>

In [2]:
# 1. Load the training data with optimized types
file_path = r"D:\all\Python\ML Projects\New York City Taxi Trip Duration\notebook\data\train.csv"
optimized_dtypes = {
    'vendor_id': 'int8', 'passenger_count': 'int8',
    'pickup_longitude': 'float32', 'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32', 'dropoff_latitude': 'float32',
    'store_and_fwd_flag': 'category', 'trip_duration': 'int32'
}

df_train = pd.read_csv(
    file_path, 
    dtype=optimized_dtypes, 
    parse_dates=['pickup_datetime', 'dropoff_datetime']
)



# 3. Domain-Knowledge Outlier Filtering
# Keep trips that are at least 1 minute long (avoids immediate cancellations/glitches)
# Keep trips under 2 hours (7200 seconds) or 3 hours (10800 seconds) to remove physical impossibilities
valid_idx = (df_train['trip_duration'] >= 60) & (df_train['trip_duration'] <= 7200)
df_train = df_train[valid_idx].copy()

### Feature Engineering: Temporal Extraction & Data Cleaning

In [3]:
# 2. Extract temporal features
df_train['pickup_hour'] = df_train['pickup_datetime'].dt.hour
df_train['pickup_dayofweek'] = df_train['pickup_datetime'].dt.dayofweek
df_train['pickup_month'] = df_train['pickup_datetime'].dt.month

# Calculate Haversine distance (in kilometers)
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df_train['haversine_dist_km'] = haversine_distance(
    df_train['pickup_latitude'], df_train['pickup_longitude'],
    df_train['dropoff_latitude'], df_train['dropoff_longitude']
)

# 3. Drop columns that cause leakage or cannot be processed
df_train = df_train.drop(columns=['id', 'pickup_datetime', 'dropoff_datetime'])

In [4]:
df_train.head(3)

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_dayofweek,pickup_month,haversine_dist_km
0,2,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,0,3,1.498675
1,1,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,6,6,1.805410
2,2,1,-73.979027,40.763939,-74.005333,40.710087,N,2124,11,1,1,6.385065


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Getting X and Y variables</p>

In [5]:
# 4. Define Features (X) and Target (y)
X = df_train.drop(columns=['trip_duration'])

# Apply the log transformation to the target variable to optimize for RMSLE
y = np.log1p(df_train['trip_duration'])

In [6]:
X.head(3)

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_hour,pickup_dayofweek,pickup_month,haversine_dist_km
0,2,1,-73.982155,40.767937,-73.964630,40.765602,N,17,0,3,1.498675
1,1,1,-73.980415,40.738564,-73.999481,40.731152,N,0,6,6,1.805410
2,2,1,-73.979027,40.763939,-74.005333,40.710087,N,11,1,1,6.385065


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Creating Data Transformation Pipeline</p>

### Creating Pipeline with Column Transformer

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Explicitly define column groups to ensure correct processing
categorical_cols = ['vendor_id', 'store_and_fwd_flag']
numerical_cols = [
    'passenger_count', 
    'pickup_longitude', 'pickup_latitude', 
    'dropoff_longitude', 'dropoff_latitude',
    'pickup_hour', 'pickup_dayofweek', 'pickup_month',
    'haversine_dist_km' # Added distance feature
]

# 2. Numerical Pipeline: Impute missing values (if any) and scale distances/times
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())                
    ]
)

# 3. Categorical Pipeline: Impute missing values and One-Hot Encode
cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        # handle_unknown='ignore' prevents crashes if the Kaggle test set introduces a weird value
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]
)

# 4. Combine both pipelines into the master preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, numerical_cols),
        ('cat_pipeline', cat_pipeline, categorical_cols)
    ],
    remainder='drop' # Drops any columns not explicitly listed above (like ID or datetime strings if you forgot to drop them)
)

# Preview the preprocessor architecture in Jupyter
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_pipeline', ...), ('cat_pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` an

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Train Test Split</p>

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Transforming the data with pipeline created</p>

In [9]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())
preprocessor.get_feature_names_out()

array(['num_pipeline__passenger_count', 'num_pipeline__pickup_longitude',
       'num_pipeline__pickup_latitude', 'num_pipeline__dropoff_longitude',
       'num_pipeline__dropoff_latitude', 'num_pipeline__pickup_hour',
       'num_pipeline__pickup_dayofweek', 'num_pipeline__pickup_month',
       'num_pipeline__haversine_dist_km', 'cat_pipeline__vendor_id_2',
       'cat_pipeline__store_and_fwd_flag_Y'], dtype=object)

In [10]:
X_train.head(3)

,num_pipeline__passenger_count,num_pipeline__pickup_longitude,num_pipeline__pickup_latitude,num_pipeline__dropoff_longitude,num_pipeline__dropoff_latitude,num_pipeline__pickup_hour,num_pipeline__pickup_dayofweek,num_pipeline__pickup_month,num_pipeline__haversine_dist_km,cat_pipeline__vendor_id_2,cat_pipeline__store_and_fwd_flag_Y
0,2.537483,-0.016348,-0.284575,0.062966,-0.044309,-0.095032,-1.560918,0.287490,-0.568453,1.0,0.0
1,-0.505946,1.338898,0.713914,-0.128222,0.164640,1.467468,-1.560918,-0.901945,1.461400,1.0,0.0
2,-0.505946,-0.471115,-1.452901,-0.244927,-1.075876,-1.501282,1.509384,0.287490,-0.411721,0.0,0.0


In [11]:
X_test.head(3)

,num_pipeline__passenger_count,num_pipeline__pickup_longitude,num_pipeline__pickup_latitude,num_pipeline__dropoff_longitude,num_pipeline__dropoff_latitude,num_pipeline__pickup_hour,num_pipeline__pickup_dayofweek,num_pipeline__pickup_month,num_pipeline__haversine_dist_km,cat_pipeline__vendor_id_2,cat_pipeline__store_and_fwd_flag_Y
0,0.254911,0.049016,1.625082,0.028712,0.123946,1.311218,0.485950,0.287490,0.398074,1.0,0.0
1,-0.505946,-0.301087,-0.257669,-0.447069,-0.249674,1.154968,-0.025767,-0.901945,-0.597911,1.0,0.0
2,-0.505946,-0.016647,1.323361,0.384999,1.043764,-0.095032,1.509384,1.476926,-0.198757,1.0,0.0


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Model Training Baseline models</p>

### Create an Evaluate Function to give all metrics after model Training

In [12]:
def evaluate_model(true_log, predicted_log):
    # 1. Calculate stable metrics in the log-space (where the model trained)
    rmsle = root_mean_squared_error(true_log, predicted_log)
    r2_log = r2_score(true_log, predicted_log)
    
    # 2. Reverse the log transformation to get actual seconds for error tracking
    true_real = np.expm1(true_log)
    predicted_real = np.expm1(predicted_log)
    
    # 3. Calculate human-interpretable metrics in raw seconds
    mae = mean_absolute_error(true_real, predicted_real)
    rmse = root_mean_squared_error(true_real, predicted_real)
    
    return rmsle, mae, rmse, r2_log

### Training Various models

In [13]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "XGBRegressor": XGBRegressor(n_jobs=-1), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False, thread_count=-1),
    "LGB Regressor": LGBMRegressor(n_jobs=-1),
    "Random Forest Regressor": RandomForestRegressor(n_jobs=-1)

    # --- WARNING: These models are computationally extreme for 1.45M rows ---
    # "K-Neighbors Regressor": KNeighborsRegressor(),
    # "AdaBoost Regressor": AdaBoostRegressor(),
}

model_list = []
r2_list = []

# Using X_train and y_train as defined earlier (adjust if your variables are named xtrain/ytrain)
for i in range(len(list(models))):
    model = list(models.values())[i]
    model_name = list(models.keys())[i]
    
    # Train model
    model.fit(X_train, y_train) 

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_test)
    
    # Evaluate Train and Validation dataset (Unpacking all 4 metrics)
    train_rmsle, train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    val_rmsle, val_mae, val_rmse, val_r2 = evaluate_model(y_test, y_val_pred)
    
    print(model_name)
    model_list.append(model_name)
    
    print('Model performance for Training set')
    print("- Kaggle RMSLE Score: {:.4f}".format(train_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(train_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Validation set')
    print("- Kaggle RMSLE Score: {:.4f}".format(val_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(val_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(val_mae))
    print("- R2 Score: {:.4f}".format(val_r2))
    
    r2_list.append(val_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Kaggle RMSLE Score: 0.5655
- Root Mean Squared Error (seconds): 3218649304279004498487151334118129664.0000
- Mean Absolute Error (seconds): 2990750023282604254631951553527808.0000
- R2 Score: 0.3967
----------------------------------
Model performance for Validation set
- Kaggle RMSLE Score: 0.6098
- Root Mean Squared Error (seconds): 11491267730276034095939296171080479815358084479372319484647833600.0000
- Mean Absolute Error (seconds): 21354954932816723681939482483702397237764125024564702687002624.0000
- R2 Score: 0.2983


Lasso
Model performance for Training set
- Kaggle RMSLE Score: 0.7280
- Root Mean Squared Error (seconds): 680.7156
- Mean Absolute Error (seconds): 442.9788
- R2 Score: 0.0000
----------------------------------
Model performance for Validation set
- Kaggle RMSLE Score: 0.7279
- Root Mean Squared Error (seconds): 678.4008
- Mean Absolute Error (seconds): 441.8920
- R2 Score: -0.0000


Ridge
Model performance for

### Results

In [ ]:
df_results = pd.DataFrame(list(zip(model_list, r2_list)), 
 columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)
df_results

,Model Name,R2_Score
7,Random Forest Regressor,0.798766
5,CatBoosting Regressor,0.797265
4,XGBRegressor,0.789818
6,LGB Regressor,0.767807
3,Decision Tree,0.584087
2,Ridge,0.298280
0,Linear Regression,0.298280
1,Lasso,-0.000005


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Hyperparameter tuning</p>

### Definition to print evaluated model results

In [15]:
def print_evaluated_results(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate Train and Test dataset (unpacking all 4 metrics)
    train_rmsle, train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_rmsle, test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    # Printing results
    print('Model performance for Training set')
    print("- Kaggle RMSLE Score: {:.4f}".format(train_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(train_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Kaggle RMSLE Score: {:.4f}".format(test_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(test_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))

### Tuning Catboost

In [ ]:
# 1. Initialize CatBoost with Early Stopping
cbr = CatBoostRegressor(
    iterations=4000,           
    od_type='Iter',            
    od_wait=50,                
    thread_count=-1,
    random_seed=42,
    verbose=False
)

# 2. Narrowed parameter space centered around known optimal regions
param_dist_cb = {
    'depth': [7, 8],               # Narrowed down to the best depths
    'learning_rate': [0.08, 0.1],  # Kept to the higher performing rates
    'l2_leaf_reg': [3, 5, 7]       # Tightened regularization bounds
}

# 3. Instantiate RandomizedSearchCV object
fit_params = {
    "eval_set": [(X_test, y_test)],
    "verbose": False
}

rscv_cb = RandomizedSearchCV(
    estimator=cbr, 
    param_distributions=param_dist_cb, 
    n_iter=5,                      # Reduced from 20 to 5 (Cuts runtime by 75%)
    scoring='neg_root_mean_squared_error', 
    cv=3,                          # 3 folds * 5 iterations = 15 fits total (instead of 60)
    n_jobs=-1, 
    random_state=42,
    verbose=2
)

# 4. Fit with validation data
print("Starting Focused CatBoost tuning...")
rscv_cb.fit(X_train, y_train, **fit_params)

# 5. Extract results
print("\n==========================================")
print("BEST CATBOOST PARAMETERS")
print("==========================================")
print(rscv_cb.best_params_)
print(f"Optimal Trees built: {rscv_cb.best_estimator_.tree_count_}")
print(f"Best CV RMSLE Score: {-rscv_cb.best_score_:.4f}")

# 6. Evaluate the best model
best_cbr = rscv_cb.best_estimator_

print("\n==========================================")
print("FINAL MODEL PERFORMANCE")
print("==========================================")
print_evaluated_results(best_cbr, X_train, y_train, X_test, y_test)

Starting Focused CatBoost tuning...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

BEST CATBOOST PARAMETERS
{'learning_rate': 0.1, 'l2_leaf_reg': 5, 'depth': 8}
Optimal Trees built: 4000
Best CV RMSLE Score: 0.3107

FINAL MODEL PERFORMANCE
Model performance for Training set
- Kaggle RMSLE Score: 0.2917
- Root Mean Squared Error (seconds): 268.7200
- Mean Absolute Error (seconds): 161.4762
- R2 Score: 0.8395
----------------------------------
Model performance for Test set
- Kaggle RMSLE Score: 0.3098
- Root Mean Squared Error (seconds): 281.3839
- Mean Absolute Error (seconds): 167.1664
- R2 Score: 0.8189


### Tuning XGBoost

In [17]:
# 1. Initialize XGBoost with Early Stopping configuration
# tree_method='hist' is crucial for datasets over 1 million rows
xgbr = XGBRegressor(
    tree_method='hist',
    objective='reg:squarederror',
    n_estimators=4000,          # Set artificially high; early stopping will cut it off
    early_stopping_rounds=50,   # Halt if validation error doesn't drop for 50 rounds
    n_jobs=-1,
    random_state=42
)

# 2. Comprehensive, regularized parameter space
param_dist_xgb = {
    'max_depth': [5, 6, 7], 
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'subsample': [0.8, 0.85, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'min_child_weight': [5, 10, 15],
    'reg_alpha': [0.5, 1.0, 2.0],
    'reg_lambda': [5, 10, 15],
    'gamma': [0, 0.1, 0.2]
}

# 3. Instantiate RandomizedSearchCV object
# Note: We pass the validation set in fit_params to enable early stopping per fold
fit_params = {
    "eval_set": [(X_test, y_test)],
    "verbose": False
}

rscv_xgb = RandomizedSearchCV(
    estimator=xgbr,
    param_distributions=param_dist_xgb,
    n_iter=20,                  # 20 combinations provides a solid search across this grid
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# 4. Fit with validation data
print("Starting Comprehensive XGBoost tuning with Early Stopping...")
rscv_xgb.fit(X_train, y_train, **fit_params)

# 5. Extract results
print("\n==========================================")
print("BEST XGBOOST PARAMETERS")
print("==========================================")
print(rscv_xgb.best_params_)
print(f"Optimal Trees built: {rscv_xgb.best_estimator_.best_iteration}")
print(f"Best CV RMSLE Score: {-rscv_xgb.best_score_:.4f}")

# 6. Evaluate the best model
best_xgbr = rscv_xgb.best_estimator_

print("\n==========================================")
print("FINAL MODEL PERFORMANCE")
print("==========================================")
print_evaluated_results(best_xgbr, X_train, y_train, X_test, y_test)

Starting Comprehensive XGBoost tuning with Early Stopping...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

BEST XGBOOST PARAMETERS
{'subsample': 0.9, 'reg_lambda': 10, 'reg_alpha': 0.5, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}
Optimal Trees built: 3998
Best CV RMSLE Score: 0.3099

FINAL MODEL PERFORMANCE
Model performance for Training set
- Kaggle RMSLE Score: 0.2756
- Root Mean Squared Error (seconds): 261.1467
- Mean Absolute Error (seconds): 156.2877
- R2 Score: 0.8567
----------------------------------
Model performance for Test set
- Kaggle RMSLE Score: 0.3088
- Root Mean Squared Error (seconds): 280.2209
- Mean Absolute Error (seconds): 166.3102
- R2 Score: 0.8200


### Tuning LightGBM

In [18]:
# 1. Initialize LightGBM
lgbr = LGBMRegressor(
    objective='regression',
    n_estimators=4000,          # Set artificially high; early stopping will cut it off
    n_jobs=-1,
    random_state=42,
    verbose=-1                  # Suppresses LightGBM's excessive C++ console warnings
)

# 2. Comprehensive, regularized parameter space
# Note: LightGBM controls complexity primarily through num_leaves, not max_depth
param_dist_lgb = {
    'num_leaves': [31, 63, 127],       # Maximum tree leaves (equivalent to depth 5, 6, 7)
    'max_depth': [-1, 7, 9],           # -1 allows num_leaves to strictly govern depth
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'subsample': [0.8, 0.85, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'min_child_samples': [20, 50, 100],# Minimum data per leaf to prevent outlier mapping
    'reg_alpha': [0.5, 1.0, 2.0],
    'reg_lambda': [5, 10, 15]
}

# 3. Instantiate RandomizedSearchCV object
# Pass validation set and early stopping callback to fit_params
fit_params = {
    "eval_set": [(X_test, y_test)],
    "callbacks": [early_stopping(stopping_rounds=50, verbose=False)]
}

rscv_lgb = RandomizedSearchCV(
    estimator=lgbr,
    param_distributions=param_dist_lgb,
    n_iter=20,                  
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# 4. Fit with validation data
print("Starting Comprehensive LightGBM tuning with Early Stopping...")
rscv_lgb.fit(X_train, y_train, **fit_params)

# 5. Extract results
print("\n==========================================")
print("BEST LIGHTGBM PARAMETERS")
print("==========================================")
print(rscv_lgb.best_params_)
print(f"Optimal Trees built: {rscv_lgb.best_estimator_.best_iteration_}")
print(f"Best CV RMSLE Score: {-rscv_lgb.best_score_:.4f}")

# 6. Evaluate the best model
best_lgbr = rscv_lgb.best_estimator_

print("\n==========================================")
print("FINAL MODEL PERFORMANCE")
print("==========================================")
print_evaluated_results(best_lgbr, X_train, y_train, X_test, y_test)

Starting Comprehensive LightGBM tuning with Early Stopping...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

BEST LIGHTGBM PARAMETERS
{'subsample': 1.0, 'reg_lambda': 15, 'reg_alpha': 2.0, 'num_leaves': 127, 'min_child_samples': 50, 'max_depth': 9, 'learning_rate': 0.08, 'colsample_bytree': 1.0}
Optimal Trees built: 4000
Best CV RMSLE Score: 0.3059

FINAL MODEL PERFORMANCE
Model performance for Training set
- Kaggle RMSLE Score: 0.2601
- Root Mean Squared Error (seconds): 250.4961
- Mean Absolute Error (seconds): 147.8787
- R2 Score: 0.8724
----------------------------------
Model performance for Test set
- Kaggle RMSLE Score: 0.3034
- Root Mean Squared Error (seconds): 277.5440
- Mean Absolute Error (seconds): 163.8193
- R2 Score: 0.8263


### Tuning Random Forest

In [19]:
# 1. Initialize Random Forest
# Early stopping does not apply to parallel ensemble methods
rfr = RandomForestRegressor(
    n_jobs=-1,
    random_state=42
)

# 2. Regularized parameter space optimized for large datasets
param_dist_rf = {
    'n_estimators': [100, 200, 300],          # RF requires significantly fewer trees than boosting
    'max_depth': [10, 15, 20, None],          # Deeper bounds than boosting, but capped to prevent memory exhaustion
    'min_samples_split': [10, 20, 50],        # Minimum samples required to split an internal node
    'min_samples_leaf': [10, 20, 50],         # Minimum samples required to be at a leaf node
    'max_features': ['sqrt', 'log2', 0.5]     # Limits features considered per split to drastically reduce training time
}

# 3. Instantiate RandomizedSearchCV object
rscv_rf = RandomizedSearchCV(
    estimator=rfr,
    param_distributions=param_dist_rf,
    n_iter=10,                                # Capped at 10 due to severe computational cost per fit
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# 4. Fit the model 
print("Starting Comprehensive Random Forest tuning...")
print("Note: This will take significantly longer than XGBoost or LightGBM.")
rscv_rf.fit(X_train, y_train)

# 5. Extract results
print("\n==========================================")
print("BEST RANDOM FOREST PARAMETERS")
print("==========================================")
print(rscv_rf.best_params_)
print(f"Best CV RMSLE Score: {-rscv_rf.best_score_:.4f}")

# 6. Evaluate the best model
best_rfr = rscv_rf.best_estimator_

print("\n==========================================")
print("FINAL MODEL PERFORMANCE")
print("==========================================")
print_evaluated_results(best_rfr, X_train, y_train, X_test, y_test)

Starting Comprehensive Random Forest tuning...
Note: This will take significantly longer than XGBoost or LightGBM.
Fitting 3 folds for each of 10 candidates, totalling 30 fits

BEST RANDOM FOREST PARAMETERS
{'n_estimators': 100, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 20}
Best CV RMSLE Score: 0.3352

FINAL MODEL PERFORMANCE
Model performance for Training set
- Kaggle RMSLE Score: 0.2976
- Root Mean Squared Error (seconds): 271.2861
- Mean Absolute Error (seconds): 162.6243
- R2 Score: 0.8329
----------------------------------
Model performance for Test set
- Kaggle RMSLE Score: 0.3334
- Root Mean Squared Error (seconds): 299.3768
- Mean Absolute Error (seconds): 181.5630
- R2 Score: 0.7902


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Voting Regressor</p>

In [20]:
# 1. Initialize the Voting Regressor with your tuned models
# We assign slightly higher weights (2) to the gradient boosting models 
# as they typically outperform Random Forest on spatial tabular data.
ensemble_model = VotingRegressor(
    estimators=[
        ('xgb', best_xgbr),
        ('lgb', best_lgbr),
        ('cat', best_cbr),
        ('rf', best_rfr)
    ],
    weights=[2, 2, 2, 1] 
)

# 2. Fit the ensemble model
print("Fitting the Voting Regressor...")
print("Note: This will take a significant amount of time as it must re-fit all 4 models from scratch.")
ensemble_model.fit(X_train, y_train)

# 3. Evaluate the final ensemble
print("\n==========================================")
print("FINAL ENSEMBLE MODEL PERFORMANCE")
print("==========================================")
print_evaluated_results(ensemble_model, X_train, y_train, X_test, y_test)

Fitting the Voting Regressor...
Note: This will take a significant amount of time as it must re-fit all 4 models from scratch.


ValueError: Must have at least 1 validation dataset for early stopping.

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">XGboost Model Feature Importances</p>

In [ ]:
# Extract feature importances from the tuned XGBoost model
feature_imp = best_xgbr.feature_importances_

# Extract feature names directly from the training dataset
feature_nm = X_train.columns

# Bind importances to feature names
imp_series = pd.Series(feature_imp, index=feature_nm)

# Print the numerical values sorted highest to lowest
print("XGBoost Feature Importances:")
print(imp_series.sort_values(ascending=False))
print('\n')

# Generate the horizontal bar chart
imp_series.sort_values().plot(
    kind='barh',
    xlabel='Feature Importance Score',
    ylabel='Feature Name',
    title='XGBoost Feature Importances',
    figsize=(10, 6) # Added figsize for better readability with long column names
)
plt.show()